# tRNA41 Dependency Map

Computes epistasis-based and single-pass dependency maps for tRNA-Ala (tRNA41) on chr1,
then compares predicted contact structure against the known RNA secondary structure.

**Three approaches compared:**
1. Epistasis metrics from double-mutant embeddings (`epi_R_singles`)
2. Log-odds dependency map (MLM head)
3. Embedding perturbation map (cosine distance)

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure genebeddings is importable (editable install or sys.path fallback)
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "genebeddings").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations, product
from scipy.stats import pearsonr
from seqmat import SeqMat

from genebeddings import VariantEmbeddingDB
from genebeddings.genebeddings import (
    add_epistasis_metrics,
    compute_logodds_dependency_map,
    compute_embedding_perturbation_map,
)

In [ ]:
# --- Model config (change these to switch models) ---
from genebeddings.wrappers import RiNALMoWrapper, SpeciesLMWrapper

# Epistasis metrics model
EPI_MODEL_CLS = RiNALMoWrapper
EPI_CONTEXT = 511
EPI_DB_PATH = 'embeddings/rinalmo.db'

# Dependency map model (needs MLM head for log-odds; any model works for embedding perturbation)
DEPMAP_MODEL_CLS = SpeciesLMWrapper
DEPMAP_CONTEXT = 600

# --- tRNA41 (tRNA-Ala) coordinates ---
CHROM = 'chr1'
START = 159141611
END = 159141684
GENE = 'tRNA41'
CHROM_NUM = '1'
STRAND = 'N'  # negative strand
MAX_DISTANCE = 100
BASES = 'ATGC'
PAD = 256  # flanking context for dependency maps

## 2. Generate all epistasis variant pairs

In [ ]:
def get_all_epistasis_ids(seq, indices, gene, chrom, zyg="N", max_distance=100):
    """Generate epistasis IDs for all position pairs within max_distance, all non-ref ALT combos."""
    items = []
    for i, j in combinations(range(len(seq)), 2):
        if j - i > max_distance:
            continue
        pos1, pos2 = int(indices[i]), int(indices[j])
        ref1, ref2 = seq[i], seq[j]
        if ref1 not in BASES or ref2 not in BASES:
            continue
        alts1 = [b for b in BASES if b != ref1]
        alts2 = [b for b in BASES if b != ref2]
        for alt1, alt2 in product(alts1, alts2):
            id1 = f"{gene}:{chrom}:{pos1}:{ref1}:{alt1}:{zyg}"
            id2 = f"{gene}:{chrom}:{pos2}:{ref2}:{alt2}:{zyg}"
            items.append({'epistasis_id': f"{id1}|{id2}", 'pos1': pos1, 'pos2': pos2})
    return pd.DataFrame(items)


s = SeqMat.from_fasta('hg38', CHROM, START, END)
df = get_all_epistasis_ids(s.seq, s.index, GENE, CHROM_NUM, STRAND, MAX_DISTANCE)
print(f"{len(df)} epistasis pairs generated")

## 3. Epistasis metrics

In [ ]:
epi_model = EPI_MODEL_CLS()
db = VariantEmbeddingDB(EPI_DB_PATH)

epi_data = add_epistasis_metrics(
    df,
    db,
    model=epi_model,
    id_col="epistasis_id",
    context=EPI_CONTEXT,
    show_progress=True,
    force=False,
)
epi_data

## 4. Dependency maps (single-pass)

In [ ]:
depmap_model = DEPMAP_MODEL_CLS()
seq_padded = SeqMat.from_fasta('hg38', CHROM, START - PAD, END + PAD).seq
tRNA_positions = list(range(PAD, PAD + END - START + 1))

result_logodds = compute_logodds_dependency_map(
    depmap_model, seq_padded,
    positions=tRNA_positions,
    show_progress=True,
)
result_logodds.plot(title=f"{DEPMAP_MODEL_CLS.__name__} log-odds dependency map")

result_embed = compute_embedding_perturbation_map(
    depmap_model, seq_padded,
    positions=tRNA_positions,
    diff="cosine",
    show_progress=True,
)
result_embed.plot(title=f"{DEPMAP_MODEL_CLS.__name__} embedding perturbation map")

## 5. RNA secondary structure ground truth

tRNA-Ala cloverleaf structure on the reverse-complement strand.

In [ ]:
def dotbracket_to_contact_map(ss):
    """Convert dot-bracket notation to a symmetric binary contact matrix."""
    opens = "([{<ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    closes = ")]}>abcdefghijklmnopqrstuvwxyz"
    N = len(ss)
    M = np.zeros((N, N), dtype=int)
    stack = {ch: [] for ch in opens}
    for i, ch in enumerate(ss):
        if ch in opens:
            stack[ch].append(i)
        elif ch in closes:
            j = stack[opens[closes.index(ch)]].pop()
            M[i, j] = M[j, i] = 1
    return M


# Dot-bracket with gaps aligned to reference (dots in refseq = gaps to strip)
ss_raw = '.>>>>>>>..>>>>..........<<<<.>>>>>.........................<<<<<................>>>>>.......<<<<<<<<<<<<.'
ss_raw = ss_raw.replace('>', '(').replace('<', ')')
refseq_raw = '.GTCTCTGTGGCGCAATGGAcgA.GCGCGCTGGACTTCTA..................ATCCAGAG...........GtTCCGGGTTCGAGTCCCGGCAGAGATG'

# Strip alignment gaps
ss = ''.join(c for c, r in zip(ss_raw, refseq_raw) if r != '.')
M_true = dotbracket_to_contact_map(ss)

plt.figure(figsize=(6, 6))
plt.imshow(M_true, cmap='viridis', origin='upper')
plt.title("tRNA41 secondary structure contact map")
plt.xlabel("Position")
plt.ylabel("Position")
plt.colorbar()
plt.show()

print(f"Structure length: {len(ss)}, base pairs: {M_true.sum() // 2}")

## 6. Epistasis heatmaps

In [ ]:
def plot_epistasis_heatmap(data, metric, title=None, fillna=False):
    """Pivot epistasis data into a symmetric heatmap."""
    dep = data.pivot_table(index='pos1', columns='pos2', values=metric, aggfunc='max')
    dep = dep.combine_first(dep.T)
    if fillna:
        dep = dep.fillna(0)
    plt.figure(figsize=(10, 8))
    sns.heatmap(dep.iloc[::-1, ::-1], cmap='magma', robust=True)
    plt.title(title or metric)
    plt.xlabel("Position")
    plt.ylabel("Position")
    plt.tight_layout()
    plt.show()
    return dep


epi_name = EPI_MODEL_CLS.__name__.replace("Wrapper", "")
dep_singles = plot_epistasis_heatmap(epi_data, 'epi_R_singles', f'{epi_name} epi_R_singles (max per pair)')
dep_mahal = plot_epistasis_heatmap(epi_data, 'epi_mahal', f'{epi_name} epi_mahal (max per pair)')

## 7. Correlation with true secondary structure

In [ ]:
def upper_tri_flatten(M):
    i, j = np.triu_indices_from(M, k=1)
    return M[i, j]


def correlate_with_structure(pred_matrix, M_true, label):
    """Compute Pearson correlation between predicted map and true contact map (upper triangle)."""
    pred = pred_matrix.copy()
    pred[np.isnan(pred)] = 0
    true_flat = upper_tri_flatten(M_true)
    pred_flat = upper_tri_flatten(pred)
    r, p = pearsonr(true_flat, pred_flat)
    print(f"{label}: Pearson r = {r:.4f}, p = {p:.2e}")
    return r, p


depmap_name = DEPMAP_MODEL_CLS.__name__.replace("Wrapper", "")

# Log-odds dependency map
correlate_with_structure(result_logodds.matrix[::-1, ::-1], M_true, f"Log-odds ({depmap_name})")

# Embedding perturbation map
correlate_with_structure(result_embed.matrix[::-1, ::-1], M_true, f"Embedding perturbation ({depmap_name})")

# Epistasis epi_R_singles
epi_matrix = dep_singles.combine_first(dep_singles.T).fillna(0).values
correlate_with_structure(epi_matrix, M_true, f"Epistasis epi_R_singles ({epi_name})")